## Data download

The aim is to synthetically create a non-gridded dataset of sea level pressure anomalies. 

To obtain sea level pressure anomalies we first need to get:
1. Climatologies of mean sea level pressure (`msl`) over the period 1979--2011 (following IceNet)
2. `msl` data for the training period

Anomalies are then calculated by subtracting the climatological mean for each calendar month from the `msl` data.

In [ ]:
# Set the base path where data should be stored. This should be the same as what you use in your IceNet-MP config.
base_path = "/LOCAL/PATH/WHERE/YOU/WANT/TO/STORE/DATA/"

In [ ]:
import os
from pathlib import Path

data_path = (Path(base_path) / "data" / "notebooks" / "extract_anomalies").resolve()
data_path.mkdir(parents=True, exist_ok=True)
current_directory = Path.cwd()

### Download monthly means (1979-2011)

We'll download from the CDS dataset [ERA5 monthly averaged data on single levels from 1940 to present](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means?tab=download). We'll use the CDS API directly, as it is not currently clear how to do so with Anemoi. To use the CDS API you need to have the `cdsapi` package installed and have a `.cdsapirc` file set up under your `$HOME`.

Download only data for the Southern hemisphere. You can do the same for the Northern hemisphere by changing the `area` key to `[90, -180, 0, 180]`, or leave out the key altogether to get global data. The data is saved in `msl_monthly_south_1979_2020.nc`. Provide the full path if you want it saved under another directory, e.g. `../data/msl_monthly_south_1979_2020.nc`.

In [ ]:
import cdsapi

dataset = "reanalysis-era5-single-levels-monthly-means"
request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": ["mean_sea_level_pressure"],
    "year": [str(year) for year in range(1979, 2021)],
    "month": [str(month) for month in range(1, 13)],
    "time": ["00:00"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [0, -180, -90, 180],
}

download_path = data_path / "msl" / "msl_monthly_south_1979_2020.nc"
download_path.parent.mkdir(parents=True, exist_ok=True)
if download_path.is_file():
    print(f"{download_path} already exists")
else:
    client = cdsapi.Client()
    client.retrieve(dataset, request).download(download_path)

### Download daily data (2014-2020)

Now download the daily `msl` data from ERA5. This data could be downloaded using Anemoi, but it would be in a different resolution and gridding system (o320) than the data downloaded directly using the CDS API, which means you'll have to regrid it. You can call the command from this notebook with the following:

In [ ]:
import cdsapi

dataset = "reanalysis-era5-single-levels"
client = cdsapi.Client()

for year in range(2014, 2021):
    for month in range(1, 13):
        request = {
            "product_type": ["reanalysis"],
            "variable": ["mean_sea_level_pressure"],
            "year": [f"{year}"],
            "month": [f"{month:02d}"],
            "day": [f"{d:02d}" for d in range(1, 32)],
            "time": ["12:00"],
            "data_format": "netcdf",
            "download_format": "unarchived",
            "grid": [0.25, 0.25],
            "area": [0, -180, -90, 180],
        }
        download_path = data_path / "msl" / f"msl_daily_south_{year}{month:02d}.nc"
        download_path.parent.mkdir(parents=True, exist_ok=True)
        if download_path.is_file():
            print(f"{download_path} already exists")
        else:
            client.retrieve(dataset, request).download(download_path)
os.chdir(current_directory)

This will save the daily data group into files by month at `${data_path}/data/msl/msl_daily_south_YYYYMM.nc`.

## Data checks

### Daily data

In [ ]:
import xarray as xr

ds = xr.open_dataset(data_path / "msl" / "msl_daily_south_201405.nc")
print(ds)
lat_step = abs(ds.latitude[1].item() - ds.latitude[0].item())
lon_step = abs(ds.longitude[1].item() - ds.longitude[0].item())
print(f"Resolution: {lat_step} x {lon_step}")

In [ ]:
import xarray as xr

# Open all files
ds = xr.open_mfdataset((data_path / "msl").glob("msl_daily_south_*.nc"))
print(ds)

# Take the mean across years for each calendar month. Shape: (12, lat, lon)
clim = ds["msl"].groupby("valid_time.month").mean(dim="valid_time")
len(clim)

In [ ]:
clim.to_dataset(name="msl").to_netcdf(data_path / "climatology_12_month.nc")

## Calculate anomalies

In [ ]:
clim = xr.open_dataset(data_path / "climatology_12_month.nc")
clim

In [ ]:
import xarray as xr
import glob
import os

# open clim file if not already loaded from previous
clim = xr.open_dataset(data_path / "climatology_12_month.nc")

# Daily files folder
input_folder = data_path / "msl"
output_folder = data_path / "msl_anom"

# Create output folder if it doesn't exist
output_folder.mkdir(parents=True, exist_ok=True)

# Iterate over daily .nc files
for daily_file in sorted(Path(data_path / "msl").glob("msl_daily_south_*.nc")):
    print(f"Processing {daily_file}")
    ds_daily = xr.open_dataset(daily_file)

    # Extract month from the first time value
    month = ds_daily["valid_time"].dt.month[0].item()

    # Select the corresponding climatology slice
    clim_month = clim.sel(month=month)

    # Subtract climatology from daily values
    anomaly = ds_daily["msl"] - clim_month["msl"]

    # Save anomaly under new folder with original filename
    output_file = output_folder / daily_file.name
    anomaly.to_dataset(name="msl_anom").to_netcdf(output_file)

    print(f"Saved {output_file}")
    ds_daily.close()
    
print("Finished anomaly calculation")

## Visualize

In [ ]:
ds_anom = xr.open_dataset(data_path / "msl_anom" / "msl_daily_south_201401.nc")
ds_msl = xr.open_dataset(data_path / "msl" / "msl_daily_south_201401.nc")

ds_msl_anom = ds_anom["msl_anom"].isel(valid_time=0) / 100  # Convert Pa → hPa
ds_msl = ds_msl["msl"].isel(valid_time=0) / 100  # Convert Pa → hPa

In [ ]:
antarctic_extent = [-180, 180, -90, -60]
arctic_extent = [-180, 180, 60, 90]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean


def plot_msl(msl_data, title):
    is_arctic = msl_data.latitude.mean().item() > 0
    proj = ccrs.NorthPolarStereo() if is_arctic else ccrs.SouthPolarStereo()

    plt.figure(figsize=(8, 8))
    ax = plt.axes(projection=proj)

    # Set appropriate extent for polar views
    if is_arctic:
        ax.set_extent(arctic_extent, crs=ccrs.PlateCarree())
    else:
        ax.set_extent(antarctic_extent, crs=ccrs.PlateCarree())

    # Basemap features
    ax.coastlines(resolution="50m", linewidth=1)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    # ax.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5)

    # Pressure levels
    levels = np.linspace(msl_data.min().item(), msl_data.max().item(), 30)
    cs = ax.contourf(
        msl_data.longitude,
        msl_data.latitude,
        msl_data,
        levels=levels,
        transform=ccrs.PlateCarree(),
        cmap=cmocean.cm.diff,
        extend="both",
    )

    # Colorbar
    plt.colorbar(cs, orientation="horizontal", pad=0.05, aspect=50, label="MSLP (hPa)")

    ax.set_title(title)

    # Apply tight layout
    plt.tight_layout()
    plt.show()

In [ ]:
plot_msl(ds_msl, title="MSLP (Antarctic)")
plot_msl(ds_msl_anom, title="MSLP Anomalies (Antarctic)")